# TabDDPM on the Credit Dataset

This notebook implements **TabDDPM** on the **Credit** dataset from the UCI Machine Learning Repository.

## Data loading

In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
default_of_credit_card_clients = fetch_ucirepo(id=350) 
  
# data (as pandas dataframes) 
X = default_of_credit_card_clients.data.features 
y = default_of_credit_card_clients.data.targets 

df = pd.concat([X, y], axis=1)

In [2]:
df.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
for col in df.select_dtypes(include='object').columns:
    bad = (df[col] != df[col].astype(str).str.strip()).sum()
    if bad > 0:
        print(col, bad)

The object columns does not contain leading spaces in any rows.
This dataset does not contain any missing values.

In [4]:
print(df.duplicated().sum())

35


The dataset contains 35 duplicate rows. These were retained, since identical feature vectors may correspond to different clients and are not necessarily accidental errors.

## Column groups

The columns are separated into categorical and numerical features.

In [5]:
categorical_columns = ['X2', 'X3', 'X4', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'Y']

numerical_columns = ['X1', 'X5', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23']

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['Y']
)

## Train-test split

The dataset is split into **80% training data** and **20% test data**.  
`random_state=42` ensures reproducibility, and `stratify=df['Y']` preserves the class distribution of the target in both splits.

## Quick check

Before training, it is useful to confirm the split sizes, verify that no missing values remain in the categorical columns, and check the class balance in the target column.

In [7]:
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('\nRemaining missing values:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nTarget distribution in full data:')
print(df['Y'].value_counts(normalize=True))
print('\nTarget distribution in train data:')
print(train_df['Y'].value_counts(normalize=True))
print('\nTarget distribution in test data:')
print(test_df['Y'].value_counts(normalize=True))

Train shape: (24000, 24)
Test shape: (6000, 24)

Remaining missing values:
Series([], dtype: int64)

Target distribution in full data:
Y
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Target distribution in train data:
Y
0    0.778792
1    0.221208
Name: proportion, dtype: float64

Target distribution in test data:
Y
0    0.778833
1    0.221167
Name: proportion, dtype: float64


## Summary

So far, the notebook:
- loads the Credit dataset from `ucimlrepo`
- merges features and target into one DataFrame
- checks hidden leading and trailing spaces from categorical values
- keeps duplicate rows
- defines categorical and numerical column groups
- creates a stratified 80/20 train-test split

## Preprocessing plan

Before training TabDDPM, numerical columns are standardized, while categorical columns are one-hot encoded. The preprocessing steps are fitted on the training data only and then applied to both splits, so the model receives a fully numeric representation without leaking information from the test set.